import библиотек

In [2]:
!pip install catboost

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.1/97.1 MB 8.9 MB/s eta 0:00:00


In [3]:
from pathlib import Path
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score
from catboost import CatBoostClassifier

In [4]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [5]:
train_path = Path('/content/drive/MyDrive/aichallenge/firstsecond/train.csv')
test_path = Path('/content/drive/MyDrive/aichallenge/firstsecond/test.csv')
output_path = Path('/content/drive/MyDrive/aichallenge/firstsecond/submission.csv')

ID_COL = 'claim_id'
target = 'is_valid'

drop_cols = [
    target,
    ID_COL,
]
train = pd.read_csv(train_path, low_memory=False)
test = pd.read_csv(test_path, low_memory=False)

feature enginereeng

In [6]:
tr = train.copy(); tr["where"] = 'train'
te = test.copy(); te["where"] = 'test'
df = pd.concat([te, tr], ignore_index=True)
print(df.shape)
df["claim_dt"] = pd.to_datetime(df['first_event_time'])
df['content_dt'] = pd.to_datetime(df['content_registered_time'])
df["content_age_hour"] = (df["claim_dt"] - df["content_dt"]).dt.total_seconds() / 3600
df["claim_hour"] = df["claim_dt"].dt.hour
df["dayofweek"] = df["claim_dt"].dt.dayofweek
df["dayofweek"] = df["dayofweek"].map({0:0, 1:0, 2:0, 3:0, 4:0, 5:1, 6:1})
df = df.sort_values("claim_dt")
df["content_prev_claim"] = df.groupby("id_content").cumcount()
df["owner_prev_claim"] = df.groupby("id_content_owner").cumcount()
df["hours_since_last_claim_post"] = df.groupby("id_content")["claim_dt"].diff().dt.total_seconds() / 3600
df["hours_since_last_claim_owner"] = df.groupby("id_content_owner")["claim_dt"].diff().dt.total_seconds() / 3600

(56104, 39)


Настройка датасета

In [7]:
X_train = df[df["where"] == "train"].copy()
X_test = df[df["where"] == 'test'].copy()
y_train = X_train[target]
drop = ['where', 'claim_id', 'first_event_time', 'content_registered_time',
        'id_content', 'id_content_owner', 'claim_dt', 'content_dt']
X_train = X_train.drop(drop + [target], axis=1)
X_test = X_test.drop(drop, axis=1 )
cat_cols = [c for c in X_train.columns if X_train[c].dtype == "object"]
cat_cols += ['registered_phone_country_id', 'mobile_phone_country_id',
             'profile_country_id', 'ip_country_id',
             'claim_user_registered_phone_country_id', 'claim_user_profile_country_id',
             'registered_year', 'claim_user_registered_year', 'age_bucket',
             'friends_bucket','claim_user_age_bucket', 'claim_user_friends_bucket']

In [ ]:
X_tr, X_val, y_tr, y_val = train_test_split(
    X_train, y_train, test_size=0.2, stratify=y_train, random_state=42
)

In [9]:
model = CatBoostClassifier(
  iterations=5000,
  learning_rate=0.01,
  depth=8,
  l2_leaf_reg=3.0,
  min_data_in_leaf=10,       # было 20 — меньше лист, больше гибкости
  auto_class_weights='Balanced',
  eval_metric='F1',
  early_stopping_rounds=300,
  random_seed=42,
  verbose=100,
)

In [10]:
model.fit(X_tr, y_tr, cat_features = cat_cols, eval_set=(X_val, y_val))

0:	learn: 0.6907469	test: 0.6784606	best: 0.6784606 (0)	total: 395ms	remaining: 32m 54s
100:	learn: 0.7250885	test: 0.6840745	best: 0.7127362 (2)	total: 26.3s	remaining: 21m 15s
200:	learn: 0.7390895	test: 0.6892342	best: 0.7127362 (2)	total: 43.7s	remaining: 17m 23s
300:	learn: 0.7463267	test: 0.6887485	best: 0.7127362 (2)	total: 1m	remaining: 15m 36s
Stopped by overfitting detector  (300 iterations wait)

bestTest = 0.7127362183
bestIteration = 2

Shrink model to first 3 iterations.


CatBoostClassifier(auto_class_weights='Balanced', depth=8, early_stopping_rounds=300, eval_metric='F1', iterations=5000, l2_leaf_reg=3.0, learning_rate=0.01, min_data_in_leaf=10, random_seed=42, verbose=100)

In [11]:
drop = ['where', 'claim_id', 'first_event_time', 'content_registered_time',
        'id_content', 'id_content_owner', 'claim_dt', 'content_dt']

X_test = df[df['where'] == 'test'].drop(columns=drop)

In [12]:
print(model.is_fitted())   # False — значит, модель пустая, нужно обучить

True


In [13]:
# 1. Подбор лучшего порога на валидации
val_probs = model.predict_proba(X_val)[:, 1]
best_t, best_f1 = 0.5, 0.0

for t in np.arange(0.1, 0.9, 0.01):
    current_f1 = f1_score(y_val, (val_probs >= t).astype(int))
    if current_f1 > best_f1:
        best_f1, best_t = current_f1, t

print(f'🎯 Лучший порог: {best_t:.2f} | F1 на валидации: {best_f1:.4f}')

# 2. Предсказание на тесте (X_test уже с 39 правильными колонками)
test_probs = model.predict_proba(X_test)[:, 1]
predictions = (test_probs >= best_t).astype(int)

# 3. Собираем сабмишен
#    ID берём из df с тем же фильтром, чтобы порядок гарантированно совпал с X_test
test_ids = df[df['where'] == 'test']['claim_id'].values

submission = pd.DataFrame({
    'claim_id': test_ids,
    'is_valid': predictions
})

# 4. Сохраняем (локально + на Google Диск)
output_path = '/content/drive/MyDrive/aichallenge/firstsecond/submission.csv'
submission.to_csv('/content/submission.csv', index=False)
submission.to_csv(output_path, index=False)

# 5. Авто-скачивание файла в браузер
from google.colab import files
files.download('/content/submission.csv')

# 6. Sanity checks
print('\n✅ Сабмишен сохранён!')
print('Баланс классов:')
print(submission['is_valid'].value_counts())
print('\nПервые 10 строк:')
print(submission.head(10))

🎯 Лучший порог: 0.50 | F1 на валидации: 0.3480


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


✅ Сабмишен сохранён!
Баланс классов:
is_valid
1    3897
0    3549
Name: count, dtype: int64

Первые 10 строк:
                                            claim_id  is_valid
0  6f7960dd60ef18c96ac31dfcb7d96c5d2c874df750d2b7...         1
1  95ba2a3c946db8e412d583235dea20ad3a2629cab5856b...         0
2  bb0af7bf46f39eee745a09e8d3ed2ae1c5c4b3939820ee...         1
3  85f55f738fcfcefb2570321c610bb2d4d40eb8f7fe4ebc...         1
4  99c0083e402ccb55c0ed358ec6b5344c7785192c985ba4...         0
5  2b58bb8d601c0af249492aef72704568d249c5a2823760...         0
6  a0444524c12e0e16b6b541513f92f404c037031d3beec6...         0
7  94d88dc3314fe941148fae63ab4f7efd0c5748e31f7e82...         0
8  bbe045f60bab026383dd63ea7e00aa8465ffe3ce1360ec...         1
9  5ad9f3ff52b55011888f033336aa6bed6487880053908b...         1
